# **CODE 8b: XGBoost MODEL TRAINING (Nifty 100)**

Trains an XGBoost conviction classifier using walk-forward validation, a financial-utility scoring function, inverse-frequency class weights, and a hybrid RandomOverSampler (search) + SMOTENC (final) resampling strategy.

---

## Input files (from Google Drive)
| File | Source | Purpose |
|------|--------|---------|
| `train_data_nifty100.parquet` | Code 8a | Cleaned train split (2011–2019) + `date`, `symbol`, `conviction_label` |
| `feature_metadata_nifty100.csv` | Code 8a | Feature list (model features only) |
| `feature_reference.csv` | project | Drives data-quality checks; `xgb_relevant='No'` features ignored |
| `features_to_prune.csv` *(optional)* | you | One column `feature_name`; features to drop on a re-run |

## Output files (auto-downloaded to your Downloads folder)
| File | Purpose |
|------|---------|
| `best_model.pkl` | Trained XGBoost model |
| `label_mapping.pkl` | Conviction→numeric mapping (Code 8c loads this) |
| `categorical_encoders.pkl` | Label encoders for categorical features |
| `feature_columns_model.pkl` | Exact feature list & order used (Code 8c loads this) |
| `feature_importance.csv` | Feature importance ranking |
| `data_quality_before.csv` | NaN / inf / blank / dtype per column BEFORE cleaning |
| `data_quality_after.csv` | NaN / inf / blank / dtype per column AFTER cleaning |
| `cleaning_report.csv` | **Column-wise before/after NaN+inf+blank, cleaning method, fill value** |
| `training_results.csv` | Per-iteration search summary |
| `cv_fold_metrics.csv` | Per-fold metrics |
| `best_hyperparameters.txt` | Winning hyperparameters |

---

## Key methodological points
- **No scaling, no cyclical encoding** — XGBoost handles raw magnitudes; only NaN handling & label-encoding of categoricals is done. This differs from the LSTM pipeline.
- **`n_estimators` is a tuned hyperparameter** (no early stopping) — the final model uses exactly the tree count that was cross-validated. See note in Step 7.
- **Walk-forward validation:** Train≤2016→Val 2017, ≤2017→2018, ≤2018→2019.
- **Conviction label mapping is fixed and printed** — must match Code 8c.

---

## **STEP 0: Mount Google Drive**

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
print("\n" + "="*80)
print("Google Drive mounted successfully!")
print("="*80)

Mounted at /content/drive

Google Drive mounted successfully!


## **STEP 1: Import Libraries**

In [2]:
import pandas as pd
import numpy as np
import pickle
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.model_selection import ParameterSampler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
from imblearn.over_sampling import RandomOverSampler   # fast — used in search
from imblearn.over_sampling import SMOTENC             # quality — used for final model

print("="*80)
print("CODE 8b: XGBoost MODEL TRAINING")
print("="*80)
print(f"\n✓ Libraries imported  |  XGBoost {xgb.__version__}")
print(f"  Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

CODE 8b: XGBoost MODEL TRAINING

✓ Libraries imported  |  XGBoost 3.3.0
  Time: 2026-08-16 18:55:40



## **STEP 2: Configuration**

**⚠️ Update `DRIVE_FOLDER`.** Review oversampling targets and the label mapping printed below.

In [3]:
# ── UPDATE THIS PATH ────────────────────────────────────────────────────────
DRIVE_FOLDER = '/content/drive/MyDrive/masters/'

SUFFIX = 'midcap150'   # stock universe (matches Code 8a output suffix)

# Input files (Google Drive)
TRAIN_FILE     = os.path.join(DRIVE_FOLDER, f'train_data_{SUFFIX}.parquet')
METADATA_FILE  = os.path.join(DRIVE_FOLDER, f'feature_metadata_{SUFFIX}.csv')
FEATURE_REF_FILE = os.path.join(DRIVE_FOLDER, 'feature_reference.csv')

# ── Inter-stock feature toggle (for ablation: with vs without inter-stock) ──
# When True, drops every feature where feature_reference inter_stock_dependency == 'Yes'
# (cluster, counter-cluster, commodity, relative-strength-vs-sector, etc.).
# Run once with False and once with True to compare model performance.
DROP_INTER_STOCK = True          #False   # ← set True to exclude inter-stock features

# Optional prune file (set to a path on a re-run; None on first run)
PRUNE_FILE = os.path.join(DRIVE_FOLDER, 'features_to_prune.csv') # e.g. None   # e.g. os.path.join(DRIVE_FOLDER, 'features_to_prune.csv')

# Output folder (Colab local, auto-downloaded at the end)
OUTPUT_FOLDER = '/content/model_outputs/'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ── Search configuration ────────────────────────────────────────────────────
# ── Per-combination outputs (Requests 1 & 2) ───────────────────────────────
SAVE_COMBO_MODELS = True   # Req1: save each combo's full-data final model
COLLECT_GAIN      = True   # Req2: per-feature gain for every fold × combo

N_ITER       = 500     # hyperparameter combinations sampled from the grid
RANDOM_STATE = 42

# ── Conviction label mapping (MUST match Code 8c) ───────────────────────────
LABEL_MAPPING_CONFIG = {'Ignore': 0, 'Low': 1, 'Medium': 2, 'High': 3}  # matches Code 7

# ── Oversampling targets per class (None = leave untouched) ─────────────────
# DISABLED: class imbalance is handled by (a) tuned class weights and (b) the
# expected-utility decision rule. Oversampling machinery kept intact but dormant.
OVERSAMPLE_TARGETS = {
    0: None,   # Ignore  — untouched
    1: None,   # Low     — untouched
    2: None,   # Medium  — untouched
    3: None,   # High    — untouched
}

print("-"*80)
print("CONFIGURATION")
print("-"*80)
print(f"  Universe       : {SUFFIX}")
print(f"  Train file     : {TRAIN_FILE}")
print(f"  Search iters   : {N_ITER}")
print(f"  Prune file     : {PRUNE_FILE if PRUNE_FILE else 'None (first run)'}")

print("\nConviction label → numeric mapping (MUST match Code 8c):")
for lbl, num in sorted(LABEL_MAPPING_CONFIG.items(), key=lambda x: x[1]):
    print(f"    {num}  =  {lbl}")

print("\nOversampling targets (None = untouched):")
for cls, tgt in OVERSAMPLE_TARGETS.items():
    lbl = [k for k,v in LABEL_MAPPING_CONFIG.items() if v == cls][0]
    print(f"    Class {cls} ({lbl:8s}): {'untouched' if tgt is None else f'target {tgt:.0%}'}")
print()

for fpath, name in [(TRAIN_FILE,'train parquet'), (METADATA_FILE,'metadata'),
                    (FEATURE_REF_FILE,'feature_reference')]:
    if not os.path.exists(fpath):
        raise FileNotFoundError(f"{name} not found: {fpath}")
    print(f"  ✓ {name}")
print()

--------------------------------------------------------------------------------
CONFIGURATION
--------------------------------------------------------------------------------
  Universe       : midcap150
  Train file     : /content/drive/MyDrive/masters/train_data_midcap150.parquet
  Search iters   : 500
  Prune file     : /content/drive/MyDrive/masters/features_to_prune.csv

Conviction label → numeric mapping (MUST match Code 8c):
    0  =  Ignore
    1  =  Low
    2  =  Medium
    3  =  High

Oversampling targets (None = untouched):
    Class 0 (Ignore  ): untouched
    Class 1 (Low     ): untouched
    Class 2 (Medium  ): untouched
    Class 3 (High    ): untouched

  ✓ train parquet
  ✓ metadata
  ✓ feature_reference



## **STEP 3: Load Data**

In [4]:
print("-"*80)
print("LOADING DATA")
print("-"*80)

train_df    = pd.read_parquet(TRAIN_FILE)
metadata_df = pd.read_csv(METADATA_FILE)
feature_ref = pd.read_csv(FEATURE_REF_FILE)

print(f"✓ train_data    : {len(train_df):,} rows, {len(train_df.columns)} columns")
print(f"✓ feature_meta  : {len(metadata_df)} features")
print(f"✓ feature_ref   : {len(feature_ref)} catalogued features")

feature_columns_all = metadata_df['feature_name'].tolist()
print(f"\nFeatures from metadata: {len(feature_columns_all)}")
print()

--------------------------------------------------------------------------------
LOADING DATA
--------------------------------------------------------------------------------
✓ train_data    : 286,739 rows, 278 columns
✓ feature_meta  : 275 features
✓ feature_ref   : 347 catalogued features

Features from metadata: 275



## **STEP 4: Data Quality Report — BEFORE cleaning**

Per-column NaN count, blank/empty count, and dtype, saved to `data_quality_before.csv`.

In [5]:
print("-"*80)
print("DATA QUALITY REPORT — BEFORE CLEANING  (train split, 2011-2019)")
print("-"*80)

def data_quality_report(df, cols):
    """Per-column NaN, inf, blank, dtype, unique. inf counted for numeric cols."""
    rows = []
    n = len(df)
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c]
        nan_ct = int(s.isnull().sum())
        if s.dtype.kind in 'fiu':
            inf_ct   = int(np.isinf(s.to_numpy(dtype='float64', na_value=np.nan)).sum())
            blank_ct = 0
        else:
            inf_ct   = 0
            blank_ct = int((s.astype(str).str.strip() == '').sum())
        rows.append({
            'feature'     : c,
            'dtype'       : str(s.dtype),
            'n_nan'       : nan_ct,
            'n_inf'       : inf_ct,
            'n_blank'     : blank_ct,
            'pct_missing' : round((nan_ct + inf_ct) / n * 100, 3),
            'n_unique'    : int(s.nunique(dropna=True)),
        })
    return pd.DataFrame(rows).sort_values('pct_missing', ascending=False)

dq_before = data_quality_report(train_df, feature_columns_all)
dq_before.to_csv(os.path.join(OUTPUT_FOLDER, 'data_quality_before.csv'), index=False)

print(f"Columns with NaN  : {(dq_before['n_nan']>0).sum()} / {len(dq_before)}")
print(f"Columns with inf  : {(dq_before['n_inf']>0).sum()} / {len(dq_before)}")
print(f"Total NaN cells   : {dq_before['n_nan'].sum():,}")
print(f"Total inf cells   : {dq_before['n_inf'].sum():,}")
print("\nTop 15 columns by % missing (NaN+inf):")
print(dq_before.head(15).to_string(index=False))
print("\n✓ Saved: data_quality_before.csv")
print()

--------------------------------------------------------------------------------
DATA QUALITY REPORT — BEFORE CLEANING  (train split, 2011-2019)
--------------------------------------------------------------------------------
Columns with NaN  : 52 / 275
Columns with inf  : 0 / 275
Total NaN cells   : 12,159,206
Total inf cells   : 0

Top 15 columns by % missing (NaN+inf):
                  feature   dtype  n_nan  n_inf  n_blank  pct_missing  n_unique
         mapped_commodity  object 276421      0        0       96.402         7
      commodity_direction float32 276421      0        0       96.402         2
    commodity_correlation float32 276421      0        0       96.402        42
      commodity_return_2d float32 236226      0        0       82.384      4874
commodity_volatility_120d float32 236226      0        0       82.384      4895
 commodity_volatility_60d float32 236226      0        0       82.384      4899
 commodity_volatility_20d float32 236226      0        0       8

## **STEP 5: Feature Selection + Per-Feature NaN Handling**

Drops identifiers and `xgb_relevant='No'` features, applies any prune list, then handles residual NaNs using `feature_reference.csv` guidance (mean/mode/zero/forward-fill). XGBoost-specific: no scaling, no cyclical encoding.

In [6]:
print("-"*80)
print("FEATURE SELECTION")
print("-"*80)

# Drop identifiers — never model features (carried only for folds/sim)
IDENTIFIERS = {'date','Date','DATE','symbol','Symbol','ticker','Ticker'}
feature_columns = [c for c in feature_columns_all if c not in IDENTIFIERS]
dropped_ids = [c for c in feature_columns_all if c in IDENTIFIERS]
if dropped_ids:
    print(f"  Dropped identifiers: {dropped_ids}")

# Drop xgb_relevant == 'No'
xgb_no = set(feature_ref.loc[feature_ref['xgb_relevant'] == 'No', 'feature_name'])
dropped_xgb = [c for c in feature_columns if c in xgb_no]
feature_columns = [c for c in feature_columns if c not in xgb_no]
print(f"  Dropped xgb_relevant='No': {len(dropped_xgb)} features")

# Inter-stock toggle: drop features with inter_stock_dependency == 'Yes'
if DROP_INTER_STOCK:
    inter_yes = set(feature_ref.loc[
        feature_ref['inter_stock_dependency'].astype(str).str.strip().str.lower() == 'yes',
        'feature_name'])
    dropped_inter = [c for c in feature_columns if c in inter_yes]
    feature_columns = [c for c in feature_columns if c not in inter_yes]
    print(f"  DROP_INTER_STOCK=True → dropped {len(dropped_inter)} inter-stock features")
    print(f"    e.g. {dropped_inter[:8]}{' ...' if len(dropped_inter) > 8 else ''}")
else:
    n_inter = int((feature_ref['inter_stock_dependency'].astype(str)
                   .str.strip().str.lower() == 'yes').sum())
    print(f"  DROP_INTER_STOCK=False → keeping inter-stock features "
          f"({n_inter} such features in reference)")

# Optional prune list
print("\nFeature pruning:")
if PRUNE_FILE and os.path.exists(PRUNE_FILE):
    prune_list = pd.read_csv(PRUNE_FILE)['feature_name'].dropna().tolist()
    found     = [c for c in prune_list if c in feature_columns]
    not_found = [c for c in prune_list if c not in feature_columns]
    feature_columns = [c for c in feature_columns if c not in found]
    print(f"  {'Feature':<40} {'In Dataset?':<12} Action")
    print(f"  {'-'*40} {'-'*12} {'-'*8}")
    for f in found:     print(f"  {f:<40} {'Yes':<12} PRUNED")
    for f in not_found: print(f"  {f:<40} {'NOT FOUND':<12} skipped")
    print(f"\n  Pruned: {len(found)}  |  not found: {len(not_found)}")
elif PRUNE_FILE:
    print(f"  ⚠️  Prune file not found: {PRUNE_FILE} — no pruning")
else:
    print(f"  PRUNE_FILE = None — no pruning (first run)")

print(f"\n  Final feature count: {len(feature_columns)}")
print()

# ── Extract X / y ───────────────────────────────────────────────────────────
X_train = train_df[feature_columns].copy()
y_train = train_df['conviction_label'].copy()
print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")

# ── NaN / inf handling (data already cleaned in Code 8a; this is a safety net) ──
# Strategy column is 'nan_handling' in feature_reference:
#   drop_row     : already applied in 8a — any residual NaN row is dropped here too
#   forward_fill : already filled per-stock in 8a
#   keep_nan     : NaN intentionally retained — XGBoost handles numeric NaN natively;
#                  categorical keep_nan cols get an explicit 'Missing' category
print("\nNaN / inf handling (safety net — main cleaning done in Code 8a)...")
nan_strategy = dict(zip(feature_ref['feature_name'], feature_ref['nan_handling']))

def resolve_strategy(col):
    raw = str(nan_strategy.get(col, 'forward_fill'))
    if 'drop_row' in raw:     return 'drop_row'
    if 'keep_nan' in raw:     return 'keep_nan'
    if 'forward_fill' in raw: return 'forward_fill'
    return 'forward_fill'

col_method = {c: resolve_strategy(c) for c in feature_columns}

# Convert inf/-inf → NaN (XGBoost rejects inf, accepts NaN)
_num = X_train.select_dtypes(include=[np.number]).columns
_n_inf = int(np.isinf(X_train[_num].to_numpy(dtype='float64', na_value=np.nan)).sum())
if _n_inf > 0:
    X_train[_num] = X_train[_num].replace([np.inf, -np.inf], np.nan)
    print(f"  Converted {_n_inf:,} inf/-inf → NaN")

# drop_row safety net: drop rows still NaN in any drop_row feature
drop_cols = [c for c in feature_columns if col_method[c] == 'drop_row']
if drop_cols:
    before = len(X_train)
    keep_mask = X_train[drop_cols].notnull().all(axis=1)
    X_train = X_train[keep_mask]
    y_train = y_train[keep_mask]
    if before - len(X_train) > 0:
        print(f"  drop_row safety net: removed {before-len(X_train):,} rows "
              f"with NaN in {len(drop_cols)} drop_row columns")

# forward_fill safety net: per-stock fill if any residual (no inter-stock data)
ff_cols = [c for c in feature_columns if col_method[c] == 'forward_fill']
ff_resid = [c for c in ff_cols if X_train[c].isnull().any()]
if ff_resid:
    print(f"  forward_fill safety net: {len(ff_resid)} columns had residual NaN")
    # No date/ticker index inside X_train; these are start-of-history gaps.
    # For numeric, fill with column median as a last resort (rare).
    for c in ff_resid:
        if X_train[c].dtype.kind in 'fiu':
            X_train[c] = X_train[c].fillna(X_train[c].median())

# keep_nan: leave numeric NaN as-is; give categoricals an explicit category
keep_cols = [c for c in feature_columns if col_method[c] == 'keep_nan']
print(f"  keep_nan columns: {len(keep_cols)} (numeric NaN retained for XGBoost)")
print(f"  Residual NaN in features (mostly intentional keep_nan): "
      f"{X_train[feature_columns].isnull().sum().sum():,}")
print()


--------------------------------------------------------------------------------
FEATURE SELECTION
--------------------------------------------------------------------------------
  Dropped xgb_relevant='No': 0 features
  DROP_INTER_STOCK=True → dropped 152 inter-stock features
    e.g. ['Excess_Ret_N500_1d', 'Excess_Ret_N500_2d', 'Excess_Ret_N500_3d', 'Excess_Ret_N500_5d', 'Excess_Ret_N500_10d', 'Excess_Ret_N500_20d', 'Excess_Ret_N500_60d', 'Excess_Ret_N500_120d'] ...

Feature pruning:
  Feature                                  In Dataset?  Action
  ---------------------------------------- ------------ --------
  Return_5d                                Yes          PRUNED
  Sortino_20d                              Yes          PRUNED
  Return_20d                               Yes          PRUNED
  Sharpe_20d                               Yes          PRUNED
  Return_60d                               Yes          PRUNED
  Days_Since_52W_High                      Yes          PRUNED
  

## **STEP 6: Data Quality Report — AFTER cleaning + Encode Categoricals + Label Mapping**

Confirms no NaNs remain, label-encodes categorical features (XGB needs numeric), and shows the conviction mapping and class weights.

In [7]:
print("-"*80)
print("DATA QUALITY REPORT — AFTER CLEANING  (train split, 2011-2019)")
print("-"*80)
dq_after = data_quality_report(X_train, feature_columns)
dq_after.to_csv(os.path.join(OUTPUT_FOLDER, 'data_quality_after.csv'), index=False)

# ── Merge BEFORE + AFTER + cleaning method into one column-wise report ──────
# 'col_method' was recorded during the residual NaN handling step (Step 5)
method_map = col_method      # recorded in Step 5

merged = dq_before.merge(
    dq_after, on='feature', how='right', suffixes=('_before', '_after'))
merged['cleaning_method'] = merged['feature'].map(
    lambda c: method_map.get(c, 'none'))

# Tidy column order
report_cols = ['feature', 'dtype_after', 'cleaning_method',
               'n_nan_before', 'n_inf_before', 'n_blank_before', 'pct_missing_before',
               'n_nan_after', 'n_inf_after', 'n_blank_after', 'pct_missing_after']
report_cols = [c for c in report_cols if c in merged.columns]
cleaning_report = merged[report_cols].sort_values('pct_missing_before', ascending=False)
cleaning_report.to_csv(os.path.join(OUTPUT_FOLDER, 'cleaning_report.csv'), index=False)

print(f"Columns with NaN after : {(dq_after['n_nan']>0).sum()} / {len(dq_after)}")
print(f"Columns with inf after : {(dq_after['n_inf']>0).sum()} / {len(dq_after)}")
print(f"Total NaN after        : {dq_after['n_nan'].sum():,}")
print(f"Total inf after        : {dq_after['n_inf'].sum():,}")

residual = cleaning_report[(cleaning_report.get('n_nan_after', 0) > 0) |
                           (cleaning_report.get('n_inf_after', 0) > 0)]
if len(residual) > 0:
    print(f"\n⚠️  {len(residual)} columns still have NaN/inf after cleaning:")
    print(residual[['feature','cleaning_method','n_nan_after','n_inf_after']].to_string(index=False))
else:
    print("\n✓ No residual NaN/inf in any feature after cleaning")

print("\n✓ Saved: data_quality_after.csv, cleaning_report.csv (column-wise before/after + method)")
print()

# ── Encode categorical features (label encoding — XGB needs numeric) ────────
print("Encoding categorical features (no scaling / no embedding for XGB)...")
categorical_columns = X_train.select_dtypes(include=['object','category']).columns.tolist()
label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].fillna('Unknown').astype(str))
    label_encoders[col] = le
    print(f"  {col:30s} → {len(le.classes_)} categories")
categorical_indices = [feature_columns.index(c) for c in categorical_columns]
print(f"  Categorical indices (for SMOTENC): {categorical_indices}")

non_num = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
if non_num:
    raise ValueError(f"Non-numeric columns remain: {non_num}")
print("✓ All features numeric")

# ── Label mapping + numeric target ──────────────────────────────────────────
label_mapping = LABEL_MAPPING_CONFIG
y_train_numeric = y_train.map(label_mapping)

print("\nConviction label mapping (saved for Code 8c):")
for lbl, num in sorted(label_mapping.items(), key=lambda x:x[1]):
    cnt = (y_train == lbl).sum()
    print(f"    {num} = {lbl:8s}  ({cnt:,} rows, {cnt/len(y_train)*100:.2f}%)")

# ── BASE class weights: FLAT (all 1.0) ──────────────────────────────────────
# Rationale: inverse-frequency weighting distorts predict_proba (it trains the
# model as if every class were 25% likely). The expected-utility decision rule
# needs HONEST probabilities, so cost asymmetry is applied at DECISION time via
# the utility matrix, not at training time. The only training-time tilt is the
# tuned HIGH/MEDIUM weight multipliers (see the hyperparameter grid), which exist
# to correct residual miscalibration — not to encode economics.
n_cls  = len(label_mapping)
total  = len(y_train_numeric)
counts = y_train_numeric.value_counts().to_dict()
CLASS_WEIGHTS = {c: 1.0 for c in range(n_cls)}   # base; multipliers applied per-combo

print("\nBase class weights (FLAT — multipliers tuned per combination):")
for c in range(n_cls):
    lbl = [k for k,v in label_mapping.items() if v==c][0]
    print(f"    {lbl:8s} ({c}): base weight={CLASS_WEIGHTS[c]:.3f}  "
          f"(natural share {counts.get(c,0)/total*100:5.2f}%)")
print()

--------------------------------------------------------------------------------
DATA QUALITY REPORT — AFTER CLEANING  (train split, 2011-2019)
--------------------------------------------------------------------------------
Columns with NaN after : 0 / 112
Columns with inf after : 0 / 112
Total NaN after        : 0
Total inf after        : 0

✓ No residual NaN/inf in any feature after cleaning

✓ Saved: data_quality_after.csv, cleaning_report.csv (column-wise before/after + method)

Encoding categorical features (no scaling / no embedding for XGB)...
  Categorical indices (for SMOTENC): []
✓ All features numeric

Conviction label mapping (saved for Code 8c):
    0 = Ignore    (185,004 rows, 64.52%)
    1 = Low       (13,802 rows, 4.81%)
    2 = Medium    (24,027 rows, 8.38%)
    3 = High      (63,906 rows, 22.29%)

Base class weights (FLAT — multipliers tuned per combination):
    Ignore   (0): base weight=1.000  (natural share 64.52%)
    Low      (1): base weight=1.000  (natural sha

## **STEP 7: Scoring Function & Hyperparameter Grid**

**`n_estimators` is part of the grid and early stopping is NOT used** — the final model uses exactly the tree count that was cross-validated, so search and final training are consistent and the validation fold is used only for scoring (no leakage).

In [8]:
print("-"*80)
print("SCORING FUNCTION + HYPERPARAMETER GRID")
print("-"*80)

# ── LOCKED UTILITY MATRIX ───────────────────────────────────────────────────
# rows = ACTUAL, cols = PREDICTED ; order Ignore0 Low1 Medium2 High3 (Code 7 order)
# DERIVED (not hand-picked) from 4 parameters, constrained to satisfy 3 properties:
#   1. Diagonal dominance — the correct call is the best cell in every row.
#   2. Base-rate default  — with no information the EU rule outputs IGNORE
#                           (a conviction call must be earned by evidence).
#   3. Monotonicity       — EU decisions yield High > Medium > Low > Ignore
#                           bucket returns in 12/12 simulation runs.
UTILITY_MATRIX = np.array([
# Pred: Ignore  Low  Medium  High
    [    2,    -15,   -30,   -46],   # Actual Ignore
    [    0,     10,   -11,   -22],   # Actual Low
    [   -5,      5,    30,   -11],   # Actual Medium
    [  -15,      6,    16,    50],   # Actual High
], dtype=float)

def financial_utility_score(y_true, y_pred):
    U = UTILITY_MATRIX
    return sum(U[t,p] for t,p in zip(y_true, y_pred)) / len(y_true)

def eu_decision(proba, U=None):
    """Expected-utility decision rule.
    proba: (n, n_cls) from predict_proba, columns in class-index order.
    Returns the prediction maximising sum_i P(actual=i) * U(i, j)."""
    if U is None: U = UTILITY_MATRIX
    return (np.asarray(proba) @ U).argmax(axis=1)

# Expected return per ACTUAL class (used only for the monotonicity diagnostic)
LABEL_EXPECTED_RETURN = np.array([-20.0, 10.0, 25.0, 50.0])   # Ignore, Low, Medium, High

def monotonicity_check(y_true, y_pred):
    """Mean label-implied return per PREDICTED bucket; flag High>Medium>Low>Ignore."""
    y_true=np.asarray(y_true); y_pred=np.asarray(y_pred)
    means={}
    for j in range(len(LABEL_EXPECTED_RETURN)):
        m = y_pred==j
        means[j] = float(LABEL_EXPECTED_RETURN[y_true[m]].mean()) if m.sum() else np.nan
    ok = (means[3]>means[2]>means[1]>means[0]) if not any(
         np.isnan(v) for v in means.values()) else False
    return ok, {'ret_pred_Ignore':means[0],'ret_pred_Low':means[1],
                'ret_pred_Medium':means[2],'ret_pred_High':means[3]}

def check_constraints(y_true, y_pred):
    # High is now class 3
    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    hp = cm[3,3]/cm[:,3].sum() if cm[:,3].sum()>0 else 0   # High precision
    ig = cm[0,3]/cm[0,:].sum() if cm[0,:].sum()>0 else 0   # Ignore→High
    lh = cm[1,3]/cm[1,:].sum() if cm[1,:].sum()>0 else 0   # Low→High
    # REPORT-ONLY: no longer a pass/fail gate. The utility matrix already prices
    # false Highs (-30 for Ignore->High), so a hard precision gate would fight it.
    # 'met' is retained for backward compatibility but always True.
    met = True
    return met, {'high_precision':hp,'ignore_to_high_rate':ig,'low_to_high_rate':lh}

# n_estimators is TUNED (no early stopping) — see step note
param_grid = {
    'n_estimators'     : [500, 700, 900],
    'max_depth'        : [4, 5, 6, 7, 8, 10, 12, 14, 16],
    'learning_rate'    : [0.1],              #[0.07, 0.1, 0.2],
    'min_child_weight' : [2],                #[2, 3, 5],
    'gamma'            : [0],                #[0.1, 0.2, 0.3],
    'subsample'        : [0.8],              #[0.6, 0.8, 1.0],
    'colsample_bytree' : [0.8],              #[0.6, 0.8, 1.0],
    'reg_alpha'        : [0],                #[0.1, 0.5, 1.0],
    'reg_lambda'       : [2],                #[0.5, 1.0, 1.5],
    # Tuned class-weight multipliers (base weights are flat 1.0).
    # 1.0 is included so "no tilt" can win on merit.
    'HIGH_WEIGHT_MULT'   : [0.75, 1.0, 1.5, 2.0],
    'MEDIUM_WEIGHT_MULT' : [1]               #[1.0, 1.5, 2.0],
}
# Keys that are NOT XGBoost params — stripped before constructing the model
NON_XGB_KEYS = ['HIGH_WEIGHT_MULT', 'MEDIUM_WEIGHT_MULT']

def weights_from_combo(cp):
    """Per-combination class weights: flat base, tuned High/Medium multipliers."""
    w = dict(CLASS_WEIGHTS)                      # {0:1,1:1,2:1,3:1}
    w[2] = w[2] * cp.get('MEDIUM_WEIGHT_MULT', 1.0)
    w[3] = w[3] * cp.get('HIGH_WEIGHT_MULT', 1.0)
    return w

def xgb_params(cp):
    """Strip non-XGBoost keys from a sampled combination."""
    return {k:v for k,v in cp.items() if k not in NON_XGB_KEYS}
param_combinations = list(ParameterSampler(param_grid, n_iter=N_ITER,
                                           random_state=RANDOM_STATE))

fixed_params = {
    'objective'   : 'multi:softprob',   # real probabilities for Code 8c thresholds
    'num_class'   : n_cls,
    'tree_method' : 'hist',
    'device'      : 'cuda',             # ← ADD THIS LINE (uses GPU)
    'random_state': RANDOM_STATE,
    'n_jobs'      : -1,
}

print("Tuned hyperparameters (sampled from grid):")
for p, v in param_grid.items():
    print(f"  {p:18s}: {v}")
print(f"\n  Sampling {N_ITER} combinations")
print(f"\nFixed: {fixed_params}")
print("\nNote: n_estimators is tuned and early stopping is OFF — the final")
print("      model uses exactly the cross-validated tree count.")
print()

--------------------------------------------------------------------------------
SCORING FUNCTION + HYPERPARAMETER GRID
--------------------------------------------------------------------------------
Tuned hyperparameters (sampled from grid):
  n_estimators      : [500, 700, 900]
  max_depth         : [4, 5, 6, 7, 8, 10, 12, 14, 16]
  learning_rate     : [0.1]
  min_child_weight  : [2]
  gamma             : [0]
  subsample         : [0.8]
  colsample_bytree  : [0.8]
  reg_alpha         : [0]
  reg_lambda        : [2]
  HIGH_WEIGHT_MULT  : [0.75, 1.0, 1.5, 2.0]
  MEDIUM_WEIGHT_MULT: [1]

  Sampling 500 combinations

Fixed: {'objective': 'multi:softprob', 'num_class': 4, 'tree_method': 'hist', 'device': 'cuda', 'random_state': 42, 'n_jobs': -1}

Note: n_estimators is tuned and early stopping is OFF — the final
      model uses exactly the cross-validated tree count.



## **STEP 8: Walk-Forward Folds + Resampling Plan**

In [9]:
print("-"*80)
print("WALK-FORWARD FOLDS")
print("-"*80)
date_col = next((c for c in ['date','Date','DATE'] if c in train_df.columns), None)
if date_col is None:
    raise ValueError("No date column in train_df — check Code 8a output")
dates = pd.to_datetime(train_df[date_col])

fold_config = [
    ('2011-01-01','2016-12-31','2017-01-01','2017-12-31'),
    ('2011-01-01','2017-12-31','2018-01-01','2018-12-31'),
    ('2011-01-01','2018-12-31','2019-01-01','2019-12-31'),
]
wf_folds = []
for i,(ts,te,vs,ve) in enumerate(fold_config,1):
    tr = np.where((dates>=ts)&(dates<=te))[0]
    vl = np.where((dates>=vs)&(dates<=ve))[0]
    wf_folds.append((tr,vl))
    print(f"  Fold {i}: Train {ts[:4]}–{te[:4]} ({len(tr):,})  →  Val {vs[:4]} ({len(vl):,})")
N_CV_FOLDS = len(wf_folds)

print("\nResampling plan:")
total_samples = len(y_train_numeric)
current_counts = {c:int((y_train_numeric==c).sum()) for c in range(n_cls)}
classes_to_oversample = {}
for cls, tgt in OVERSAMPLE_TARGETS.items():
    if tgt is None: continue
    tn = int(total_samples*tgt)
    if tn > current_counts[cls]:
        classes_to_oversample[cls] = tn
for cls in range(n_cls):
    lbl = [k for k,v in label_mapping.items() if v==cls][0]
    cur = current_counts[cls]
    if cls in classes_to_oversample:
        act = f"oversample → {classes_to_oversample[cls]:,}"
    else:
        act = "untouched"
    print(f"  {lbl:8s} ({cls}): {cur:,} ({cur/total_samples*100:.1f}%)  {act}")
print()

--------------------------------------------------------------------------------
WALK-FORWARD FOLDS
--------------------------------------------------------------------------------
  Fold 1: Train 2011–2016 (182,013)  →  Val 2017 (33,102)
  Fold 2: Train 2011–2017 (215,115)  →  Val 2018 (35,174)
  Fold 3: Train 2011–2018 (250,289)  →  Val 2019 (36,450)

Resampling plan:
  Ignore   (0): 185,004 (64.5%)  untouched
  Low      (1): 13,802 (4.8%)  untouched
  Medium   (2): 24,027 (8.4%)  untouched
  High     (3): 63,906 (22.3%)  untouched



## **STEP 9: Hyperparameter Search (walk-forward, RandomOverSampler)**

In [10]:
print("="*80)
print("HYPERPARAMETER SEARCH")
print("="*80)

search_results=[]; fold_results=[]
gain_records=[]                 # Req2: (feature, fold, iteration) → gain
combo_model_paths=[]            # Req1: saved per-combo model paths
best_score=-np.inf; best_params=None; best_metrics=None
start=time.time()

# Folder for per-combo models (downloaded as a zip at the end)
COMBO_DIR = os.path.join(OUTPUT_FOLDER, 'combo_models')
os.makedirs(COMBO_DIR, exist_ok=True)

#for it in range(N_ITER):
for it in range(len(param_combinations)):
    cp = param_combinations[it]
    print(f"\n{'='*80}\nITERATION {it+1}/{N_ITER}\n{'='*80}")
    print("  params:", {k:(round(v,3) if isinstance(v,float) else v) for k,v in cp.items()})

    cvs=[]; cvc=[]; chp=[]; cig=[]; clh=[]; cmono=[]
    for fn,(tri,vli) in enumerate(wf_folds,1):
        Xtr=X_train.iloc[tri].copy(); ytr=y_train_numeric.iloc[tri].copy()
        Xvl=X_train.iloc[vli].copy(); yvl=y_train_numeric.iloc[vli].copy()

        need={c:t for c,t in classes_to_oversample.items() if t>ytr.value_counts().get(c,0)}
        if need:
            Xtr,ytr=RandomOverSampler(sampling_strategy=need,
                     random_state=RANDOM_STATE+it+fn).fit_resample(Xtr,ytr)

        # No early stopping; n_estimators comes from cp.
        # xgb_params() strips the tuned weight multipliers (not XGBoost args);
        # weights_from_combo() turns them into this combo's class weights.
        combo_w = weights_from_combo(cp)
        model=xgb.XGBClassifier(**{**fixed_params,**xgb_params(cp)})
        model.fit(Xtr,ytr,sample_weight=ytr.map(combo_w),verbose=False)

        # ── Req2: per-feature GAIN for this fold (all features, zeros explicit) ──
        if COLLECT_GAIN:
            gain_map = model.get_booster().get_score(importance_type='gain')
            # XGBoost keys may be feature names or f0,f1...; map both ways
            for fi_idx, feat in enumerate(feature_columns):
                g = gain_map.get(feat, gain_map.get(f'f{fi_idx}', 0.0))
                gain_records.append({'iteration': it+1, 'fold': fn,
                                     'feature': feat, 'gain': float(g)})

        # DECISION RULE: expected utility on predict_proba (NOT argmax).
        # Selection must use the same rule Code 8c deploys.
        yp_proba=model.predict_proba(Xvl)
        yp=eu_decision(yp_proba)
        sc=financial_utility_score(yvl.values,yp)
        met,mt=check_constraints(yvl.values,yp)      # report-only
        mono_ok, mono_ret = monotonicity_check(yvl.values, yp)
        cvs.append(sc); cvc.append(met); cmono.append(mono_ok)
        chp.append(mt['high_precision']); cig.append(mt['ignore_to_high_rate']); clh.append(mt['low_to_high_rate'])
        fold_results.append({'iteration':it+1,'fold':fn,'score':sc,'meets_constraints':met,
                             'monotonic':mono_ok,**mt,**mono_ret,**cp})
        print(f"  Fold {fn}: score={sc:.2f} mono={'✓' if mono_ok else '✗'} "
              f"HighPrec={mt['high_precision']:.1%} Ign→High={mt['ignore_to_high_rate']:.1%} Low→High={mt['low_to_high_rate']:.1%}")

    ms=np.mean(cvs); allc=all(cvc)
    mhp,mig,mlh=np.mean(chp),np.mean(cig),np.mean(clh)
    print(f"  → mean score={ms:.3f}  constraints_met={'YES' if allc else 'NO'}  "
          f"avgHighPrec={mhp:.1%}")
    n_mono=int(sum(cmono))
    print(f"  → monotonic folds: {n_mono}/{len(cmono)}")
    search_results.append({'iteration':it+1,'mean_cv_score':ms,'all_constraints_met':allc,
        'monotonic_folds':n_mono,'all_folds_monotonic':n_mono==len(cmono),
        'avg_high_precision':mhp,'avg_ignore_to_high_rate':mig,'avg_low_to_high_rate':mlh,**cp})

    # ── Req1: train this combo's FULL-DATA final model and save it ─────────────
    if SAVE_COMBO_MODELS:
        need_full={c:t for c,t in classes_to_oversample.items()
                   if t>y_train_numeric.value_counts().get(c,0)}
        if need_full:
            Xf,yf=RandomOverSampler(sampling_strategy=need_full,
                   random_state=RANDOM_STATE+it).fit_resample(X_train,y_train_numeric)
        else:
            Xf,yf=X_train,y_train_numeric
        combo_model=xgb.XGBClassifier(**{**fixed_params,**xgb_params(cp)})
        combo_model.fit(Xf,yf,sample_weight=yf.map(weights_from_combo(cp)),verbose=False)
        cpath=os.path.join(COMBO_DIR, f'model_combo_{it+1:03d}.pkl')
        with open(cpath,'wb') as _f: pickle.dump(combo_model,_f)
        combo_model_paths.append({'iteration':it+1,'path':cpath,
                                  'mean_cv_score':ms,**cp})
        print(f"  💾 saved combo model: model_combo_{it+1:03d}.pkl")

    if allc and ms>best_score:
        best_score=ms; best_params=cp.copy()
        best_metrics={'avg_high_precision':mhp,'avg_ignore_to_high_rate':mig,'avg_low_to_high_rate':mlh}
        print(f"  🌟 NEW BEST (score={best_score:.3f})")

    el=time.time()-start
    print(f"  elapsed {el/60:.1f} min | ETA {el/(it+1)*(N_ITER-it-1)/60:.1f} min")

search_time=time.time()-start
print(f"\n{'='*80}\nSEARCH COMPLETE in {search_time/60:.1f} min\n{'='*80}")

if best_params is None:
    print("⚠️  No model met all constraints — selecting best by score alone")
    bi=int(np.argmax([r['mean_cv_score'] for r in search_results]))
    br=search_results[bi]
    skip={'iteration','mean_cv_score','all_constraints_met','avg_high_precision',
          'avg_ignore_to_high_rate','avg_low_to_high_rate',
          'monotonic_folds','all_folds_monotonic'}
    best_params={k:v for k,v in br.items() if k not in skip}
    best_score=br['mean_cv_score']
    best_metrics={'avg_high_precision':br['avg_high_precision'],
                  'avg_ignore_to_high_rate':br['avg_ignore_to_high_rate'],
                  'avg_low_to_high_rate':br['avg_low_to_high_rate']}

print("\nBest hyperparameters (incl. tuned n_estimators):")
for p,v in best_params.items():
    print(f"  {p:18s}: {v}")
print(f"\nBest CV score: {best_score:.4f}")
print()

HYPERPARAMETER SEARCH

ITERATION 1/500
  params: {'subsample': 0.8, 'reg_lambda': 2, 'reg_alpha': 0, 'n_estimators': 500, 'min_child_weight': 2, 'max_depth': 4, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.8, 'MEDIUM_WEIGHT_MULT': 1, 'HIGH_WEIGHT_MULT': 0.75}
  Fold 1: score=-3.80 mono=✗ HighPrec=39.6% Ign→High=4.2% Low→High=3.3%
  Fold 2: score=-1.87 mono=✗ HighPrec=17.0% Ign→High=3.0% Low→High=2.6%
  Fold 3: score=-2.81 mono=✗ HighPrec=24.8% Ign→High=2.6% Low→High=2.9%
  → mean score=-2.825  constraints_met=YES  avgHighPrec=27.1%
  → monotonic folds: 0/3
  💾 saved combo model: model_combo_001.pkl
  🌟 NEW BEST (score=-2.825)
  elapsed 0.3 min | ETA 146.2 min

ITERATION 2/500
  params: {'subsample': 0.8, 'reg_lambda': 2, 'reg_alpha': 0, 'n_estimators': 700, 'min_child_weight': 2, 'max_depth': 4, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.8, 'MEDIUM_WEIGHT_MULT': 1, 'HIGH_WEIGHT_MULT': 0.75}
  Fold 1: score=-3.80 mono=✓ HighPrec=39.1% Ign→High=5.3% Low→High=4.4%


## **STEP 10: Train Final Model (SMOTENC)**

Uses the winning hyperparameters — including the cross-validated `n_estimators` — so the deployed model is exactly the one that was validated.

In [11]:
print("="*80)
print("FINAL MODEL TRAINING")
print("="*80)
fstart=time.time()

if classes_to_oversample:
    '''
    # Comment out if there are NaN as SMOTENC can't handle
    print("Applying SMOTENC to full training data...")
    sm=SMOTENC(categorical_features=categorical_indices,
               sampling_strategy=classes_to_oversample,k_neighbors=5,
               random_state=RANDOM_STATE)
    X_res,y_res=sm.fit_resample(X_train,y_train_numeric)
    '''
    print("Applying RandomOS to full training data...")
    ros=RandomOverSampler(sampling_strategy=classes_to_oversample,
                      random_state=RANDOM_STATE)
    X_res,y_res=ros.fit_resample(X_train,y_train_numeric)
    print(f"  {len(X_train):,} → {len(X_res):,} samples")
    for c in range(n_cls):
        lbl=[k for k,v in label_mapping.items() if v==c][0]
        cnt=(y_res==c).sum()
        print(f"    {lbl:8s}: {cnt:,} ({cnt/len(y_res)*100:.1f}%)")
else:
    print("No oversampling needed — training on original data")
    X_res,y_res=X_train,y_train_numeric

print("\nTraining final model (using cross-validated n_estimators)...")
final_model=xgb.XGBClassifier(**{**fixed_params,**xgb_params(best_params)})
final_model.fit(X_res,y_res,sample_weight=y_res.map(weights_from_combo(best_params)),
                verbose=False)
print(f"  class weights used: {weights_from_combo(best_params)}")
final_time=time.time()-fstart
print(f"✓ Final model trained in {final_time/60:.1f} min "
      f"(n_estimators={best_params['n_estimators']})")
print()

FINAL MODEL TRAINING
No oversampling needed — training on original data

Training final model (using cross-validated n_estimators)...
  class weights used: {0: 1.0, 1: 1.0, 2: 1.0, 3: 0.75}
✓ Final model trained in 1.0 min (n_estimators=500)



## **STEP 11: Feature Importance**

In [12]:
print("-"*80)
print("FEATURE IMPORTANCE (top 25)")
print("-"*80)
fi=pd.DataFrame({'feature':feature_columns,'importance':final_model.feature_importances_}
    ).sort_values('importance',ascending=False)
print(fi.head(25).to_string(index=False))
fi.to_csv(os.path.join(OUTPUT_FOLDER,'feature_importance.csv'),index=False)
print("\n✓ feature_importance.csv saved")
print("\nTip: to prune, copy weak feature names into a CSV with column")
print("     'feature_name', set PRUNE_FILE in Step 2, and re-run from Step 3.")
print()

--------------------------------------------------------------------------------
FEATURE IMPORTANCE (top 25)
--------------------------------------------------------------------------------
              feature  importance
              SMA_200    0.020807
               SMA_50    0.019344
     MaxDrawdown_252d    0.019211
               SMA_20    0.018826
                  OBV    0.018807
                 Open    0.017929
      Volatility_252d    0.017220
                  Low    0.016129
        Kurtosis_252d    0.016082
                Month    0.015950
Value_Traded_MA20_Log    0.015898
               ATR_14    0.015828
            Skew_252d    0.015712
      UpDays_Pct_252d    0.015604
     MaxDrawdown_120d    0.015511
                 High    0.015262
                Close    0.014610
      Volatility_120d    0.014315
            Month_cos    0.013705
            Month_sin    0.013610
        Kurtosis_120d    0.013275
      MaxDrawdown_60d    0.013263
Lower_Lows_Count_120d    0.0

## **STEP 12: Save Artefacts**

In [13]:
print("="*80)
print("SAVING ARTEFACTS")
print("="*80)

def save_pickle(obj, name):
    p=os.path.join(OUTPUT_FOLDER,name)
    with open(p,'wb') as f: pickle.dump(obj,f)
    print(f"  ✓ {name}")
    return p

save_pickle(final_model,'best_model.pkl')
save_pickle(label_encoders,'categorical_encoders.pkl')
save_pickle(label_mapping,'label_mapping.pkl')
save_pickle(feature_columns,'feature_columns_model.pkl')
save_pickle(UTILITY_MATRIX,'utility_matrix.pkl')   # Code 8c uses this for the EU decision rule

pd.DataFrame(search_results).to_csv(os.path.join(OUTPUT_FOLDER,'training_results.csv'),index=False)
pd.DataFrame(fold_results).to_csv(os.path.join(OUTPUT_FOLDER,'cv_fold_metrics.csv'),index=False)
print("  ✓ training_results.csv, cv_fold_metrics.csv")

# ── Req2: consolidated GAIN table (feature × fold × combo) + summary ─────────
if COLLECT_GAIN and len(gain_records):
    gain_long = pd.DataFrame(gain_records)
    gain_long.to_csv(os.path.join(OUTPUT_FOLDER, 'gain_by_feature_fold_combo.csv'), index=False)

    # Wide pivot: one row per feature, one column per (combo, fold), value = gain.
    # Every feature appears (zeros are explicit because we recorded all features).
    gain_wide = gain_long.pivot_table(
        index='feature', columns=['iteration', 'fold'], values='gain', fill_value=0.0)
    gain_wide.columns = [f'combo{c}_fold{f}' for c, f in gain_wide.columns]
    gain_wide = gain_wide.reset_index()

    # Summary stats per feature to spot consistently-negligible features
    stat_cols = [c for c in gain_wide.columns if c != 'feature']
    gain_wide['gain_mean']      = gain_wide[stat_cols].mean(axis=1)
    gain_wide['gain_max']       = gain_wide[stat_cols].max(axis=1)
    gain_wide['gain_nonzero_runs'] = (gain_wide[stat_cols] > 0).sum(axis=1)
    gain_wide['total_runs']     = len(stat_cols)
    gain_wide['pct_runs_zero']  = (1 - gain_wide['gain_nonzero_runs'] / len(stat_cols)) * 100
    # Flag: never used in any tree across any fold/combo → always-zero gain
    gain_wide['always_zero']    = gain_wide['gain_nonzero_runs'] == 0
    # Order: worst (always zero, then lowest mean gain) first → easy prune scan
    gain_wide = gain_wide.sort_values(['always_zero', 'gain_mean'],
                                      ascending=[False, True])

    # Put summary columns first for readability
    front = ['feature', 'gain_mean', 'gain_max', 'gain_nonzero_runs',
             'total_runs', 'pct_runs_zero', 'always_zero']
    gain_wide = gain_wide[front + [c for c in gain_wide.columns if c not in front]]
    gain_wide.to_csv(os.path.join(OUTPUT_FOLDER, 'gain_summary_by_feature.csv'), index=False)

    n_always_zero = int(gain_wide['always_zero'].sum())
    print(f"  ✓ gain_by_feature_fold_combo.csv (long: {len(gain_long):,} rows)")
    print(f"  ✓ gain_summary_by_feature.csv (all {len(gain_wide)} features, zeros explicit)")
    print(f"    Features with ZERO gain in EVERY fold×combo: {n_always_zero} "
          f"(prime prune candidates)")

# ── Req1: shared fileset so each combo model is runnable in Code 8c ──────────
if SAVE_COMBO_MODELS and len(combo_model_paths):
    # The encoders / label_mapping / feature_columns are identical for all combos
    # (only hyperparameters differ), so one shared copy lives in combo_models/.
    for obj, nm in [(label_encoders, 'categorical_encoders.pkl'),
                    (label_mapping, 'label_mapping.pkl'),
                    (feature_columns, 'feature_columns_model.pkl'),
                    (UTILITY_MATRIX, 'utility_matrix.pkl')]:
        with open(os.path.join(COMBO_DIR, nm), 'wb') as _f:
            pickle.dump(obj, _f)
    pd.DataFrame(combo_model_paths).to_csv(
        os.path.join(COMBO_DIR, 'combo_model_index.csv'), index=False)
    print(f"  ✓ {len(combo_model_paths)} per-combo models in combo_models/ "
          f"(+ shared encoders/label_map/feature_cols + index)")

with open(os.path.join(OUTPUT_FOLDER,'best_hyperparameters.txt'),'w') as f:
    f.write("BEST HYPERPARAMETERS\n"+"="*60+"\n\n")
    f.write(f"Best CV score : {best_score:.4f}\n")
    f.write(f"Label mapping : {label_mapping}\n\n")
    for p,v in best_params.items(): f.write(f"  {p:18s}: {v}\n")
print("  ✓ best_hyperparameters.txt")

print(f"\nLabel mapping saved: {label_mapping}")
print(f"Features saved     : {len(feature_columns)}")
print()

SAVING ARTEFACTS
  ✓ best_model.pkl
  ✓ categorical_encoders.pkl
  ✓ label_mapping.pkl
  ✓ feature_columns_model.pkl
  ✓ utility_matrix.pkl
  ✓ training_results.csv, cv_fold_metrics.csv
  ✓ gain_by_feature_fold_combo.csv (long: 36,288 rows)
  ✓ gain_summary_by_feature.csv (all 112 features, zeros explicit)
    Features with ZERO gain in EVERY fold×combo: 0 (prime prune candidates)
  ✓ 108 per-combo models in combo_models/ (+ shared encoders/label_map/feature_cols + index)
  ✓ best_hyperparameters.txt

Label mapping saved: {'Ignore': 0, 'Low': 1, 'Medium': 2, 'High': 3}
Features saved     : 112



## **STEP 13: Auto-Download All Outputs**

In [14]:
print("="*80)
print("AUTO-DOWNLOADING OUTPUTS")
print("="*80)
from google.colab import files
import shutil

# List everything (including the combo_models/ subfolder) for transparency
print("Output files:")
for root, _, fs in os.walk(OUTPUT_FOLDER):
    for fn in sorted(fs):
        fp = os.path.join(root, fn)
        rel = os.path.relpath(fp, OUTPUT_FOLDER)
        print(f"  {rel:45s} ({os.path.getsize(fp)/1024:8.1f} KB)")

# Zip the WHOLE output folder (recurses into combo_models/) → one download.
# This avoids browser multi-download throttling AND includes the per-combo models.
zip_path = shutil.make_archive('/content/model_outputs_8b', 'zip', OUTPUT_FOLDER)
print(f"\n✓ Zipped all outputs ({os.path.getsize(zip_path)/(1024*1024):.1f} MB)")
files.download(zip_path)
print("✓ Downloading model_outputs_8b.zip")

print("\n"+"="*80)
print("CODE 8b COMPLETE")
print("="*80)
print("\nThe zip contains:")
print("  • best_model.pkl + categorical_encoders.pkl + label_mapping.pkl +")
print("    feature_columns_model.pkl  (the SELECTED model fileset for Code 8c)")
print("  • combo_models/  (Req1: every combo's full-data model + shared fileset")
print("    + combo_model_index.csv — each is runnable in 8c)")
print("  • gain_by_feature_fold_combo.csv + gain_summary_by_feature.csv  (Req2)")
print("  • training_results.csv, cv_fold_metrics.csv, feature_importance.csv")
print()

AUTO-DOWNLOADING OUTPUTS
Output files:
  best_hyperparameters.txt                      (     0.4 KB)
  best_model.pkl                                (128468.1 KB)
  categorical_encoders.pkl                      (     0.0 KB)
  cleaning_report.csv                           (     6.0 KB)
  cv_fold_metrics.csv                           (    64.2 KB)
  data_quality_after.csv                        (     4.1 KB)
  data_quality_before.csv                       (    11.7 KB)
  feature_columns_model.pkl                     (     1.7 KB)
  feature_importance.csv                        (     2.8 KB)
  gain_by_feature_fold_combo.csv                (  1309.2 KB)
  gain_summary_by_feature.csv                   (   669.4 KB)
  label_mapping.pkl                             (     0.1 KB)
  training_results.csv                          (    13.7 KB)
  utility_matrix.pkl                            (     0.3 KB)
  combo_models/categorical_encoders.pkl         (     0.0 KB)
  combo_models/combo_model_inde

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloading model_outputs_8b.zip

CODE 8b COMPLETE

The zip contains:
  • best_model.pkl + categorical_encoders.pkl + label_mapping.pkl +
    feature_columns_model.pkl  (the SELECTED model fileset for Code 8c)
  • combo_models/  (Req1: every combo's full-data model + shared fileset
    + combo_model_index.csv — each is runnable in 8c)
  • gain_by_feature_fold_combo.csv + gain_summary_by_feature.csv  (Req2)
  • training_results.csv, cv_fold_metrics.csv, feature_importance.csv

